In [1]:
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install sentence-transformers
!pip install modelscope

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [2]:
import torch
print("CUDA 是否可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("当前 GPU 设备:", torch.cuda.get_device_name(0))

CUDA 是否可用: True
当前 GPU 设备: NVIDIA GeForce RTX 4090


In [14]:
!pip uninstall -y unsloth unsloth_zoo
!pip install unsloth==2026.6.3

Found existing installation: unsloth 2026.7.1
Uninstalling unsloth-2026.7.1:
  Successfully uninstalled unsloth-2026.7.1
Found existing installation: unsloth_zoo 2026.7.1
Uninstalling unsloth_zoo-2026.7.1:
  Successfully uninstalled unsloth_zoo-2026.7.1
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 839.9 kB/s eta 0:00:0000:0100:03
  Using cached http://mirrors.aliyun.com/pypi/packages/45/37/d6bda1ff407500178ee4bf64dc93e8b4c5ecc5490c85f0fe272b560663b7/unsloth_zoo-2026.7.1-py3-none-any.whl (1.5 MB)


In [18]:
from unsloth import FastSentenceTransformer

In [19]:
fourbit_models = [
    "unsloth/all-MiniLM-L6-v2",
    "unsloth/embeddinggemma-300m",
    "unsloth/Qwen3-Embedding-4B",
    "unsloth/Qwen3-Embedding-0.6B",
    "unsloth/all-mpnet-base-v2",
    "unsloth/gte-modernbert-base",
    "unsloth/bge-m3"

]

In [20]:
import os; os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

model = FastSentenceTransformer.from_pretrained(
    model_name = "/root/autodl-tmp/Qwen3-Embedding-4B",
    max_seq_length = 512,   # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

==((====))==  Unsloth 2026.6.3: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [21]:
model = FastSentenceTransformer.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    task_type = "FEATURE_EXTRACTION"
)

In [23]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="/root/autodl-tmp/dataset.jsonl", split = "train")

Generating train split: 0 examples [00:00, ? examples/s]

In [24]:
dataset

Dataset({
    features: ['anchor', 'positive'],
    num_rows: 106628
})

In [25]:
print("Dataset examples:")
for i in range(6):
    print(dataset[i])

Dataset examples:
{'anchor': '.308', 'positive': 'The .308 Winchester is a popular rifle cartridge used for hunting and target shooting.'}
{'anchor': '.308', 'positive': 'Many precision rifles are chambered in .308 for its excellent long-range accuracy.'}
{'anchor': '.308', 'positive': 'The sniper selected a .308 caliber round for the mission.'}
{'anchor': '.338 lapua', 'positive': 'The .338 Lapua Magnum is a high-powered cartridge designed for extreme long-range shooting.'}
{'anchor': '.338 lapua', 'positive': 'Military snipers often use .338 Lapua for engagements beyond 1000 meters.'}
{'anchor': '.338 lapua', 'positive': 'The rifle was chambered in .338 Lapua for maximum effective range.'}


In [26]:
from sentence_transformers import util
import torch

def test_inference(model, run_name = "Run"):
    """Test model with a query and candidate sentences"""
    query = "apexification"
    candidates = [
        "a brick left by Yuki",  # Completely unrelated
        "apples are a tasty treat",  # Unrelated, but shares "ap-" prefix
        "the weed whacker uses an engine that runs on a mixture of gas and oil",  # Unrelated
        "a type of cancer treatment that uses drugs to boost the body's immune response",  # Medical context but wrong procedure
        "a plant hormone for regulating stress responses",  # Scientific but unrelated field
        "induces root tip closure in non-vital teeth"  # CORRECT - this is what apexification actually means
    ]

    with torch.inference_mode():
      with torch.autocast(device_type = "cuda", dtype = torch.float32):
          query_emb = model.encode(query, convert_to_tensor = True)
          candidate_embs = model.encode(candidates, convert_to_tensor = True)
    scores = util.cos_sim(query_emb, candidate_embs)[0]

    results = []
    for i, score in enumerate(scores):
        results.append((candidates[i], score.item()))
    results.sort(key = lambda x: x[1], reverse = True)

    print(f"\n--- {run_name} Results for query: '{query}' ---")
    for text, score in results:
        print(f"{score:.4f} | {text}")

test_inference(model, run_name = "Pre-Training")


--- Pre-Training Results for query: 'apexification' ---
0.6984 | induces root tip closure in non-vital teeth
0.4495 | a brick left by Yuki
0.4287 | apples are a tasty treat
0.3563 | a plant hormone for regulating stress responses
0.3447 | a type of cancer treatment that uses drugs to boost the body's immune response
0.2667 | the weed whacker uses an engine that runs on a mixture of gas and oil


In [37]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses
)
from sentence_transformers.training_args import BatchSamplers
from unsloth import is_bf16_supported

# This will use other positives in the same batch as negative examples
loss = losses.MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model = model,
    train_dataset = dataset,
    loss = loss,
    args = SentenceTransformerTrainingArguments(
        num_train_epochs = 1,
        # max_steps = 60,
        per_device_train_batch_size = 256,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        learning_rate = 3e-5,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 1,
        warmup_ratio = 0.03,
        report_to = "none", # Use TrackIO/WandB etc
        output_dir = "output",
        lr_scheduler_type = "constant_with_warmup",
        # Because we have duplicate anchors in the dataset, we don't want
        # to accidentally use them for negative examples
        batch_sampler = BatchSamplers.NO_DUPLICATES,
    ),

)

In [38]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 106,628 | Num Epochs = 1 | Total steps = 417
O^O/ \_/ \    Batch size per device = 256 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (256 x 1 x 1) = 256
 "-____-"     Trainable parameters = 66,060,288 of 4,087,834,624 (1.62% trained)


Step,Training Loss
1,0.209300


OutOfMemoryError: CUDA out of memory. Tried to allocate 148.00 MiB. GPU 0 has a total capacity of 23.52 GiB of which 24.69 MiB is free. Including non-PyTorch memory, this process has 23.48 GiB memory in use. Of the allocated memory 21.87 GiB is allocated by PyTorch, and 1.15 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [31]:
test_inference(model, run_name = "Post-Training")


--- Post-Training Results for query: 'apexification' ---
0.7342 | induces root tip closure in non-vital teeth
0.1242 | a brick left by Yuki
0.0718 | a plant hormone for regulating stress responses
0.0633 | a type of cancer treatment that uses drugs to boost the body's immune response
0.0081 | apples are a tasty treat
-0.0460 | the weed whacker uses an engine that runs on a mixture of gas and oil


In [32]:
model.save_pretrained("qwen_lora")  # Local saving
model.tokenizer.save_pretrained("qwen_lora")

('qwen_lora/tokenizer_config.json',
 'qwen_lora/special_tokens_map.json',
 'qwen_lora/chat_template.jinja',
 'qwen_lora/vocab.json',
 'qwen_lora/merges.txt',
 'qwen_lora/added_tokens.json',
 'qwen_lora/tokenizer.json')

In [34]:
# save to 16bit
if True:
    model.save_pretrained_merged(
        "/root/autodl-tmp/qwen_finetune_16bit", 
        tokenizer = model.tokenizer, 
        save_method = "merged_16bit",
    )

Detected local model directory: /root/autodl-tmp/Qwen3-Embedding-4B
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [00:02<00:02,  2.58s/it]

Copied model-00001-of-00002.safetensors from local model directory


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:04<00:00,  2.01s/it]


Copied model-00002-of-00002.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:12<00:00,  6.19s/it]


Unsloth: Merge process complete. Saved to `/root/autodl-tmp/qwen_finetune_16bit`


In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("qwen_finetune",)

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", quantization_method = "f16")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", quantization_method = "q4_k_m")

In [35]:
# Use new modle file 
model_finetune_16bit = FastSentenceTransformer.from_pretrained(
    model_name = "/root/autodl-tmp/qwen_finetune_16bit",
    max_seq_length = 512,   # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

==((====))==  Unsloth 2026.6.3: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [36]:
def test_inference(model_finetune_16bit, run_name = "Run"):
    """Test model with a query and candidate sentences"""
    query = "apexification"
    candidates = [
        "a brick left by Yuki",  # Completely unrelated
        "apples are a tasty treat",  # Unrelated, but shares "ap-" prefix
        "the weed whacker uses an engine that runs on a mixture of gas and oil",  # Unrelated
        "a type of cancer treatment that uses drugs to boost the body's immune response",  # Medical context but wrong procedure
        "a plant hormone for regulating stress responses",  # Scientific but unrelated field
        "induces root tip closure in non-vital teeth"  # CORRECT - this is what apexification actually means
    ]

    with torch.inference_mode():
      with torch.autocast(device_type = "cuda", dtype = torch.float32):
          query_emb = model.encode(query, convert_to_tensor = True)
          candidate_embs = model.encode(candidates, convert_to_tensor = True)
    scores = util.cos_sim(query_emb, candidate_embs)[0]

    results = []
    for i, score in enumerate(scores):
        results.append((candidates[i], score.item()))
    results.sort(key = lambda x: x[1], reverse = True)

    print(f"\n--- {run_name} Results for query: '{query}' ---")
    for text, score in results:
        print(f"{score:.4f} | {text}")

test_inference(model, run_name = "Pre-Training")


--- Pre-Training Results for query: 'apexification' ---
0.7342 | induces root tip closure in non-vital teeth
0.1242 | a brick left by Yuki
0.0718 | a plant hormone for regulating stress responses
0.0633 | a type of cancer treatment that uses drugs to boost the body's immune response
0.0081 | apples are a tasty treat
-0.0460 | the weed whacker uses an engine that runs on a mixture of gas and oil
